***RELOAD DATA & LIBRARY***

In [13]:
#Basic setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Show all columns, without hiding any
pd.set_option('display.max_columns', None)

#Set the chart background to white with a grid for easier observation and comparison of data values.
sns.set(style="whitegrid")

In [14]:
#Load the dataset
path = "..\\Data\\Processed\\data_cleaned.csv"
df = pd.read_csv(path)

#Quick look at the data
df.head()

,track_number,track_popularity,explicit,artist_popularity,artist_followers,artist_genres,album_id,album_total_tracks,album_type,track_duration_min,release_year,release_month
0,4,0,True,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.55,2025,10
1,1,0,True,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,1,single,3.07,2025,10
2,1,4,True,48,193302,Unknown,3E3zEAL8gUYWaLYB9L7gbp,1,single,2.55,2025,10
3,8,30,True,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.69,2025,10
4,2,0,True,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,2,single,2.39,2025,10


***FEATURE ENGINEERING***

Bảng tóm tắt 
new feature| description | Method/Logic ....|

In [15]:
def feature_engineering(df):
    """
    Feature Engineering cho Spotify Hit Prediction
    """

    df_eng = df.copy()

    # =========================================================
    # 1. TARGET
    # =========================================================
    df_eng['is_hit'] = (df_eng['track_popularity'] > 80).astype(int)

    # =========================================================
    # 2. DURATION FEATURES
    # =========================================================
    bins = [0, 2.5, 4.0, 20.0]
    labels = ['Short', 'Medium', 'Long']

    df_eng['duration_level'] = pd.cut(
        df_eng['track_duration_min'],
        bins=bins,
        labels=labels,
        include_lowest=True
    ).astype(str)

    # duration squared
    df_eng['track_duration_sq'] = (
        df_eng['track_duration_min'] ** 2
    )
     # short song flag
    df_eng['is_short_song'] = (
        df_eng['track_duration_min'] < 2.5
    ).astype(int)

    # =====================================================
    # 3. RELEASE DATE FEATURES
    # =====================================================

    CURRENT_YEAR = 2026

    # Convert datatype
    df_eng['release_year'] = pd.to_numeric(
        df_eng['release_year'],
        errors='coerce'
    )

    df_eng['release_month'] = pd.to_numeric(
        df_eng['release_month'],
        errors='coerce'
    )

    # Song age
    df_eng['song_age'] = (
        CURRENT_YEAR - df_eng['release_year']
    )

    # Recent song flag
    df_eng['is_recent_song'] = (
        df_eng['song_age'] <= 2
    ).astype(int)

    # Release season
    def month_to_season(month):

        if month in [12, 1, 2]:
            return 'Winter'

        elif month in [3, 4, 5]:
            return 'Spring'

        elif month in [6, 7, 8]:
            return 'Summer'

        else:
            return 'Fall'

    df_eng['release_season'] = (
        df_eng['release_month']
        .apply(month_to_season)
        .astype(str)
    )

    # =====================================================
    # 4. ARTIST FEATURES
    # =====================================================

    # Log transform followers
    df_eng['artist_followers_log'] = np.log1p(
        df_eng['artist_followers']
    )

    # Popular artist flag
    df_eng['artist_is_popular'] = (
        df_eng['artist_popularity'] >= 70
    ).astype(int)

    # Artist power score
    df_eng['artist_power_score'] = (
        df_eng['artist_popularity']
        * df_eng['artist_followers_log']
    )

    # =====================================================
    # 5. ALBUM FEATURES
    # =====================================================

    # Single album flag
    df_eng['is_single_album'] = (
        df_eng['album_total_tracks'] <= 2
    ).astype(int)

    # Lead track
    df_eng['is_lead_track'] = (
        df_eng['track_number'] == 1
    ).astype(int)

    # Relative position inside album
    df_eng['track_position_ratio'] = (
        df_eng['track_number']
        / df_eng['album_total_tracks']
    )

    # =====================================================
    # 6. GENRE FEATURES
    # =====================================================

    # Fill missing genres
    df_eng['artist_genres'] = (
        df_eng['artist_genres']
        .fillna('')
        .astype(str)
        .str.lower()
    )

    # Number of genres
    df_eng['genre_count'] = (
        df_eng['artist_genres']
        .apply(lambda x: len(x.split(',')))
    )

    # Genre keyword flags
    genre_keywords = [
        'pop',
        'rap',
        'rock',
        'hip hop',
        'dance',
        'latin'
    ]

    for genre in genre_keywords:

        col_name = (
            f'genre_{genre.replace(" ", "_")}'
        )

        df_eng[col_name] = (
            df_eng['artist_genres']
            .str.contains(genre, regex=False)
            .astype(int)
        )

    # =====================================================
    # 7. INTERACTION FEATURES
    # =====================================================

    # Artist popularity x followers
    df_eng['artist_pop_x_followers'] = (
        df_eng['artist_popularity']
        * df_eng['artist_followers_log']
    )

    # Duration x artist popularity
    df_eng['duration_x_artist_pop'] = (
        df_eng['track_duration_min']
        * df_eng['artist_popularity']
    )

    # Explicit x artist popularity
    df_eng['explicit_x_artist_pop'] = (
        df_eng['explicit'].astype(int)
        * df_eng['artist_popularity']
    )

    # =====================================================
    # 8. REMOVE LEAKAGE FEATURES
    # =====================================================

    # KHÔNG được dùng track_popularity để predict is_hit
    # vì is_hit được tạo trực tiếp từ track_popularity

    leakage_columns = [
        'track_popularity'
    ]

    df_eng = df_eng.drop(
        columns=leakage_columns,
        errors='ignore'
    )

    # =====================================================
    # FINAL INFO
    # =====================================================

    print("=" * 50)
    print("Feature Engineering Completed")
    print("=" * 50)

    print(f"Final Shape: {df_eng.shape}")

    print("\nTarget Variable:")
    print("is_hit")
    print("0 = Not Hit")
    print("1 = Hit")

    return df_eng

df_featured = feature_engineering(df)
print("df_featured created successfully.")
display(df_featured.head())

Feature Engineering Completed
Final Shape: (8582, 34)

Target Variable:
is_hit
0 = Not Hit
1 = Hit
df_featured created successfully.


,track_number,explicit,artist_popularity,artist_followers,artist_genres,album_id,album_total_tracks,album_type,track_duration_min,release_year,release_month,is_hit,duration_level,track_duration_sq,is_short_song,song_age,is_recent_song,release_season,artist_followers_log,artist_is_popular,artist_power_score,is_single_album,is_lead_track,track_position_ratio,genre_count,genre_pop,genre_rap,genre_rock,genre_hip_hop,genre_dance,genre_latin,artist_pop_x_followers,duration_x_artist_pop,explicit_x_artist_pop
0,4,True,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.55,2025,10,0,Short,2.4025,1,1,1,Fall,14.849699,1,1143.426808,0,0,0.444444,1,0,0,0,0,0,0,1143.426808,119.35,77
1,1,True,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,1,single,3.07,2025,10,0,Medium,9.4249,0,1,1,Fall,14.675628,0,939.240212,1,1,1.000000,2,0,0,0,1,0,0,939.240212,196.48,64
2,1,True,48,193302,unknown,3E3zEAL8gUYWaLYB9L7gbp,1,single,2.55,2025,10,0,Medium,6.5025,0,1,1,Fall,12.172014,0,584.256681,1,1,1.000000,1,0,0,0,0,0,0,584.256681,122.40,48
3,8,True,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.69,2025,10,0,Short,2.8561,1,1,1,Fall,14.850015,1,1143.451140,0,0,0.888889,1,0,0,0,0,0,0,1143.451140,130.13,77
4,2,True,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,2,single,2.39,2025,10,0,Short,5.7121,1,1,1,Fall,9.069122,0,435.317874,1,0,1.000000,1,0,0,0,0,0,0,435.317874,114.72,48


In [16]:
# Data saving
df_featured.to_csv(r'..\\Data\\Processed\\data_featured.csv', index=False)

### **X. Post-Checking**

In [17]:
# 1. DEFINE X AND y

target_col = 'is_hit'

y = df_featured[target_col]

drop_cols = [
    'is_hit',
    'track_popularity',
    'track_id',
    'track_name',
    'artist_name',
    'artist_genres',
    'album_name',
    'album_release_date'
]

X = df_featured.drop(columns=drop_cols, errors='ignore')

print("Feature matrix X shape:", X.shape)
print("Target y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

Feature matrix X shape: (8582, 32)
Target y shape: (8582,)

Feature columns:
['track_number', 'explicit', 'artist_popularity', 'artist_followers', 'album_id', 'album_total_tracks', 'album_type', 'track_duration_min', 'release_year', 'release_month', 'duration_level', 'track_duration_sq', 'is_short_song', 'song_age', 'is_recent_song', 'release_season', 'artist_followers_log', 'artist_is_popular', 'artist_power_score', 'is_single_album', 'is_lead_track', 'track_position_ratio', 'genre_count', 'genre_pop', 'genre_rap', 'genre_rock', 'genre_hip_hop', 'genre_dance', 'genre_latin', 'artist_pop_x_followers', 'duration_x_artist_pop', 'explicit_x_artist_pop']


In [18]:
# 2. CHECK MISSING VALUES

print("Missing values in X:")
print(X.isnull().sum()[X.isnull().sum() > 0])

print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X:
Series([], dtype: int64)

Missing values in y:
0


In [19]:
# 3. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS

numerical_cols = X.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumber of numerical columns:", len(numerical_cols))
print("Number of categorical columns:", len(categorical_cols))

Numerical columns:
['track_number', 'artist_popularity', 'artist_followers', 'album_total_tracks', 'track_duration_min', 'release_year', 'release_month', 'track_duration_sq', 'is_short_song', 'song_age', 'is_recent_song', 'artist_followers_log', 'artist_is_popular', 'artist_power_score', 'is_single_album', 'is_lead_track', 'track_position_ratio', 'genre_count', 'genre_pop', 'genre_rap', 'genre_rock', 'genre_hip_hop', 'genre_dance', 'genre_latin', 'artist_pop_x_followers', 'duration_x_artist_pop', 'explicit_x_artist_pop']

Categorical columns:
['explicit', 'album_id', 'album_type', 'duration_level', 'release_season']

Number of numerical columns: 27
Number of categorical columns: 5


C:\Users\Admin\AppData\Local\Temp\ipykernel_12828\3524250562.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(
